# Try the bike-route pipeline on your own ride video

Runs the whole pipeline on one video - every vehicle and how it relates to the rider, traffic signs typed and read,
traffic signals, lane lines, licence plates - and shows the results at the end of this page.

1. It asks for Colab's free T4 GPU. With Colab Pro, **Runtime > Change runtime type > L4** is about twice as fast.
2. Set the options in the first cell (or keep them), then **Runtime > Run all**.
3. Allow access to Google Drive when asked - results are saved there, in `My Drive/bikesafe_runs/<video>/`, as they
   are produced, so a disconnect loses nothing: run all again and it carries on where it stopped.
4. Pick your video when asked, unless you gave a Drive path in the options.

Time: see the README (it depends on the GPU and the length processed).
The models are tuned for a forward-facing handlebar camera (Insta360 single-lens 1080p exports); other cameras work,
less accurately. Large files reach Colab faster from Google Drive than through the upload button.

In [ ]:
#@title Options
DRIVE_VIDEO = ""  #@param {type:"string"}
#@markdown A video in your Google Drive, e.g. `/content/drive/MyDrive/rides/my_ride.mp4`. Empty: upload one from this computer.
START_S = 0  #@param {type:"number"}
MINUTES = 3  #@param {type:"number"}
#@markdown Process only `MINUTES` from `START_S` seconds in (0 minutes = the whole video).
CLIP_SECONDS = 60  #@param {type:"number"}
#@markdown Length of the overlay clip, from the middle of the processed part.
BRANCH = "main"  #@param {type:"string"}
REPO = "https://github.com/Robertn02/Car-detection.git"

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
assert gpu, "no GPU: Runtime > Change runtime type > GPU, then run all again"
from google.colab import drive

drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/bikesafe_runs")
RUNS.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title Install the pipeline (about 2 minutes)
ROOT = pathlib.Path("/content/Car-detection")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", BRANCH, REPO, str(ROOT)], check=True)
started = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# Colab's OpenCV builds would shadow the pinned one, and its torchaudio is built for Colab's own torch: with the pinned
# torch, transformers (which imports it when present) fails to load the depth model. The pipeline uses neither.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "opencv-python", "opencv-contrib-python",
                "opencv-python-headless", "torchaudio"])
# the versions the pipeline was validated with, torch included, so results match a local run
r = subprocess.run(["uv", "pip", "install", "--python", sys.executable, "--break-system-packages",
                    "--index-strategy", "unsafe-best-match", "-r", str(ROOT / "requirements.txt")],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
assert r.returncode == 0, "install failed (see above)"
check = "import torch, cv2; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| OpenCV', cv2.__version__)"
print(subprocess.run([sys.executable, "-c", check], capture_output=True, text=True).stdout.strip(),
      f"| installed in {time.time() - started:.0f} s")

In [ ]:
#@title Your video
INPUT = pathlib.Path("/content/input")  # read from the runtime's own disk, not through Drive
INPUT.mkdir(exist_ok=True)
if DRIVE_VIDEO:
    source = pathlib.Path(DRIVE_VIDEO)
    assert source.exists(), f"{source} not found in your Drive"
else:
    from google.colab import files

    uploaded = files.upload()
    source = pathlib.Path(next(iter(uploaded)))
suffix = f"_from{int(START_S)}s_{MINUTES:g}min" if MINUTES else ""
VIDEO = INPUT / f"{source.stem}{suffix}.mp4"
if not VIDEO.exists():
    if MINUTES:  # cut without re-encoding
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-ss", str(START_S), "-t", str(MINUTES * 60), "-i",
                        str(source), "-c", "copy", str(VIDEO)], check=True)
    else:
        shutil.copy(source, VIDEO)
OUT = RUNS / VIDEO.stem
print(f"{VIDEO.name}: {VIDEO.stat().st_size / 1e9:.2f} GB -> results in My Drive/bikesafe_runs/{VIDEO.stem}/")

In [ ]:
#@title Run the pipeline (progress is printed; the sign and lane pass is the long part)
env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-m", "bikesafe.run", str(INPUT), "--out-root", str(OUT / "work"), "--corpus",
       str(OUT / "results"), "--no-train"]  # the trained models in the repository; nothing is retrained
with subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                      bufsize=1) as p:
    for line in p.stdout:
        if line.strip() and not any(s in line for s in ("Loading weights", "HF_TOKEN", "it/s]")):
            print(line, end="", flush=True)
assert p.returncode == 0, "the pipeline stopped with an error (see above); run all again to resume"

In [ ]:
#@title Overlay clip
from IPython.display import Video, display

meta = json.loads((OUT / "work" / "perception" / VIDEO.stem / "meta.json").read_text())
length = meta["source_frames"] / meta["source_fps"]
start = max(0.0, length / 2 - CLIP_SECONDS / 2)
clip = OUT / f"overlay_{int(start)}s.mp4"
subprocess.run([sys.executable, "-m", "bikesafe.render", VIDEO.stem, "--videos", str(INPUT), "--analysis",
                str(OUT / "work" / "analysis"), "--corpus", str(OUT / "results"), "--start-s", f"{start:.1f}",
                "--duration-s", str(CLIP_SECONDS), "--out", str(clip)], cwd=ROOT, env=env, check=True)
preview = pathlib.Path("/content/preview.mp4")  # a lighter copy to play here; the full clip is on Drive
subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(clip), "-vf", "scale=960:-2", "-c:v", "libx264",
                "-crf", "28", "-pix_fmt", "yuv420p", str(preview)], check=True)
display(Video(str(preview), embed=True, width=960))
print("Box colours: red in my lane, orange other lane same way, magenta oncoming, grey parked, blue side road; "
      "'?' marks an unsure call. Signs show their type and text, signals their state.")

In [ ]:
#@title Results
import pandas as pd

R = OUT / "results" / VIDEO.stem
vehicles = pd.read_csv(R / "vehicles.csv")
counted = vehicles[vehicles.counted]
print(f"{len(vehicles)} vehicles seen, {len(counted)} came within 15 m for at least a second (counted):")
display(counted.relation_text.value_counts().rename("vehicles"))
print("Closest vehicles (one row per vehicle; 'check' says what a person should look at):")
display(vehicles.sort_values("closest_m").head(12)[["time_local", "relation_text", "confidence", "check", "closest_m",
                                                     "side", "vehicle_class", "plate"]])
timeline = pd.read_csv(R / "timeline_1s.csv")
print("Minutes by riding context:")
display(timeline.scene.value_counts().div(60).round(1).rename("minutes"))
signs = pd.read_csv(R / "signs.csv") if (R / "signs.csv").exists() else pd.DataFrame(columns=["kind"])
traffic_signs = signs[signs.kind == "sign"]
print(f"{len(traffic_signs)} traffic signs ({(traffic_signs.category != 'other_sign').mean():.0%} typed); "
      "the ones with text read:")
readable = traffic_signs.sign_text.fillna("").str.contains(r"[A-Za-z]{3}|\d{2}")  # not OCR fragments like "-" or "7"
display(traffic_signs[readable][["category", "sign_text", "speed_limit"]].head(15))
events = pd.read_csv(R / "events.csv")
print("Events:")
display(events.event.value_counts().rename("count") if len(events) else "none")
archive = shutil.make_archive(f"/content/{VIDEO.stem}_results", "zip", OUT, "results")
print(f"All tables (described in docs/OUTPUTS.md of the repository) are in My Drive/bikesafe_runs/{VIDEO.stem}/; "
      "here they are as one zip:")
from google.colab import files

files.download(archive)

In [ ]:
#@title Optional: the review page - check and correct the model's answers
#@markdown The page used to review the rides: vehicles, signs and plates, the most likely errors first.
#@markdown Decisions are saved in `My Drive/bikesafe_runs/<video>/labels/`.
review = subprocess.Popen([sys.executable, "-m", "bikesafe.review", "--analysis", str(OUT / "work" / "analysis"),
                           "--perception", str(OUT / "work" / "perception"), "--corpus", str(OUT / "results"),
                           "--videos", str(INPUT), "--labels", str(OUT / "labels")], cwd=ROOT, env=env)
time.sleep(8)
from google.colab.output import serve_kernel_port_as_window

serve_kernel_port_as_window(8765, anchor_text="Open the review page")